In [1]:


%pip -q install nibabel pydicom SimpleITK scikit-learn tqdm


Note: you may need to restart the kernel to use updated packages.


In [2]:

import os
import re
import gc
import json
import math
import time
import random
import shutil
import zipfile
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from itertools import cycle

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix

import nibabel as nib
import pydicom
import SimpleITK as sitk

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


DEVICE: cuda
GPU: Tesla T4


In [3]:


@dataclass
class CFG:
   
    KAGGLE_INPUT: str = "/kaggle/input"
    WORK_ROOT: str = "/kaggle/working/daan_adni_work"
    CACHE_ROOT: str = "/kaggle/working/daan_adni_work/cache_npy"
    CHECKPOINT_ROOT: str = "/kaggle/working/daan_adni_work/checkpoints"
    EXTRACT_ROOT: str = "/kaggle/working/daan_adni_work/extracted_zips"

    NEGATIVE_CLASS: str = "CN"
    POSITIVE_CLASS: str = "AD"
    NUM_CLASSES: int = 2

    
    INPUT_MODE: str = "MULTIMODAL"

    RUN_BOTH_DIRECTIONS: bool = True
    RUN_DIRECTION: str = "ADNI1_TO_ADNI2" 

    MAX_PAIR_GAP_DAYS: int = 180
    TEST_SIZE: float = 0.30
    VAL_FROM_TRAIN_RATIO: float = 0.30 / 0.70
    PREFILTER_BEFORE_SPLIT: bool = True
    LOAD_TIMEOUT_SEC: int = 90

    DEBUG_MAX_PAIRS_PER_DOMAIN: int | None = None

   
    TARGET_SHAPE: tuple = (96, 112, 96)
    RAW_APPROX_FOREGROUND_CROP: bool = True
    RAW_APPROX_GAUSSIAN_SIGMA: float = 0.0
    PREFER_PREPROCESSED_CANDIDATES: bool = True

    EXTRACT_ZIPS: bool = False
    FORCE_REEXTRACT: bool = False
    ZIP_NAME_FILTERS: tuple = ("adni", "mri", "pet")

    BATCH_SIZE: int = 1
    NUM_WORKERS: int = 0
    PRETRAIN_EPOCHS: int = 5
    ADAPT_EPOCHS: int = 25
    LR: float = 1e-4
    WEIGHT_DECAY: float = 1e-4
    AMP: bool = True
    GRAD_ACCUM_STEPS: int = 2
    EARLY_STOPPING_PATIENCE: int = 8
    CHECKPOINT_MONITOR: str = "BAL_ACC"  
    FEATURE_DIM: int = 256
    DAAN_GAMMA: float = 10.0
    ADV_WEIGHT: float = 1.0
    LOCAL_LOSS_WEIGHT: float = 1.0
    GLOBAL_LOSS_WEIGHT: float = 1.0
    DYNAMIC_FACTOR_INIT: float = 0.5
    USE_DYNAMIC_FACTOR: bool = True

    CLASS_LOSS: str = "weighted_ce"
    FOCAL_GAMMA: float = 2.0

    MODALITY_ALIGN_WEIGHT: float = 0.0

cfg = CFG()

WORK_ROOT = Path(cfg.WORK_ROOT)
CACHE_ROOT = Path(cfg.CACHE_ROOT)
CHECKPOINT_ROOT = Path(cfg.CHECKPOINT_ROOT)
EXTRACT_ROOT = Path(cfg.EXTRACT_ROOT)
for p in [WORK_ROOT, CACHE_ROOT, CHECKPOINT_ROOT, EXTRACT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print(json.dumps(asdict(cfg), indent=2, default=str))


{
  "KAGGLE_INPUT": "/kaggle/input",
  "WORK_ROOT": "/kaggle/working/daan_adni_work",
  "CACHE_ROOT": "/kaggle/working/daan_adni_work/cache_npy",
  "CHECKPOINT_ROOT": "/kaggle/working/daan_adni_work/checkpoints",
  "EXTRACT_ROOT": "/kaggle/working/daan_adni_work/extracted_zips",
  "NEGATIVE_CLASS": "CN",
  "POSITIVE_CLASS": "AD",
  "NUM_CLASSES": 2,
  "INPUT_MODE": "MULTIMODAL",
  "RUN_BOTH_DIRECTIONS": true,
  "RUN_DIRECTION": "ADNI1_TO_ADNI2",
  "MAX_PAIR_GAP_DAYS": 180,
  "TEST_SIZE": 0.3,
  "VAL_FROM_TRAIN_RATIO": 0.4285714285714286,
  "PREFILTER_BEFORE_SPLIT": true,
  "LOAD_TIMEOUT_SEC": 90,
  "DEBUG_MAX_PAIRS_PER_DOMAIN": null,
  "TARGET_SHAPE": [
    96,
    112,
    96
  ],
  "RAW_APPROX_FOREGROUND_CROP": true,
  "RAW_APPROX_GAUSSIAN_SIGMA": 0.0,
  "PREFER_PREPROCESSED_CANDIDATES": true,
  "EXTRACT_ZIPS": false,
  "FORCE_REEXTRACT": false,
  "ZIP_NAME_FILTERS": [
    "adni",
    "mri",
    "pet"
  ],
  "BATCH_SIZE": 1,
  "NUM_WORKERS": 0,
  "PRETRAIN_EPOCHS": 5,
  "ADAPT_EPOCHS

In [4]:


KAGGLE_INPUT = Path(cfg.KAGGLE_INPUT)

CSV_ROOT = Path("/kaggle/input/datasets/syedmuabdullah99313/rp-csvs")
THREE_DATA_ROOT = Path("/kaggle/input/datasets/wajdanrasool/adni-2-pet")
ADNI1_MRI_DATA_ROOT = Path("/kaggle/input/datasets/syedmuabdullah99313/dl-project")

CSV_FILES = {
    "ADNI1_MRI": CSV_ROOT / "improvement2_adni1_mri_5_30_2026.csv",
    "ADNI1_PET": CSV_ROOT / "ADNI_1_PET_FINAL_5_30_2026.csv",
    "ADNI2_MRI": CSV_ROOT / "improvement2_adni2_mri_5_30_2026 (1).csv",
    "ADNI2_PET": CSV_ROOT / "improvement2_adni2_pet_5_30_2026.csv",
}

EXTRACTED = {
    "ADNI1_MRI": [ADNI1_MRI_DATA_ROOT / "ADNI"],
    "ADNI1_PET": [THREE_DATA_ROOT / "improvement2_adni1_pet_clean" / "ADNI"],
    "ADNI2_MRI": [THREE_DATA_ROOT / "improvement2_adni2_mri" / "ADNI"],
    "ADNI2_PET": [THREE_DATA_ROOT / "improvement2_adni2_pet_clean" / "ADNI"],
}

ROOTS_BY_DATASET_MODALITY = {
    ("ADNI1", "MRI"): EXTRACTED["ADNI1_MRI"],
    ("ADNI1", "PET"): EXTRACTED["ADNI1_PET"],
    ("ADNI2", "MRI"): EXTRACTED["ADNI2_MRI"],
    ("ADNI2", "PET"): EXTRACTED["ADNI2_PET"],
}

EXTRACTED_ROOTS = []
for roots in EXTRACTED.values():
    for r in roots:
        r = Path(r)
        if r not in EXTRACTED_ROOTS:
            EXTRACTED_ROOTS.append(r)

print("Selected CSV files:")
for k, v in CSV_FILES.items():
    print(f"  {k}: {v} | exists={v.exists()}")

missing_csv = [k for k, v in CSV_FILES.items() if not Path(v).exists()]
if missing_csv:
    raise FileNotFoundError("Missing CSV files: " + ", ".join(missing_csv))

print("\nSelected image roots:")
for k, roots in EXTRACTED.items():
    for r in roots:
        print(f"  {k}: {r} | exists={Path(r).exists()}")

missing_roots = []
for k, roots in EXTRACTED.items():
    for r in roots:
        if not Path(r).exists():
            missing_roots.append((k, str(r)))

if missing_roots:
    print("\nMissing image roots:")
    for k, r in missing_roots:
        print(f"  {k}: {r}")
    raise FileNotFoundError("Some image roots do not exist. Check Kaggle dataset mounting paths.")

print("\nAll paths found successfully.")

Selected CSV files:
  ADNI1_MRI: /kaggle/input/datasets/syedmuabdullah99313/rp-csvs/improvement2_adni1_mri_5_30_2026.csv | exists=True
  ADNI1_PET: /kaggle/input/datasets/syedmuabdullah99313/rp-csvs/ADNI_1_PET_FINAL_5_30_2026.csv | exists=True
  ADNI2_MRI: /kaggle/input/datasets/syedmuabdullah99313/rp-csvs/improvement2_adni2_mri_5_30_2026 (1).csv | exists=True
  ADNI2_PET: /kaggle/input/datasets/syedmuabdullah99313/rp-csvs/improvement2_adni2_pet_5_30_2026.csv | exists=True

Selected image roots:
  ADNI1_MRI: /kaggle/input/datasets/syedmuabdullah99313/dl-project/ADNI | exists=True
  ADNI1_PET: /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni1_pet_clean/ADNI | exists=True
  ADNI2_MRI: /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni2_mri/ADNI | exists=True
  ADNI2_PET: /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni2_pet_clean/ADNI | exists=True

All paths found successfully.


In [5]:


def pick_col(df, candidates, required=True):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        key = cand.strip().lower()
        if key in lower_map:
            return lower_map[key]
    if required:
        raise KeyError(f"None of these columns found: {candidates}. Available: {list(df.columns)}")
    return None


def normalize_group(x):
    s = str(x).strip().upper()
    if s in {"CN", "NL", "NORMAL", "NORMAL CONTROL", "COGNITIVELY NORMAL", "CONTROL"}:
        return "CN"
    if s in {"AD", "DEMENTIA", "ALZHEIMER'S DISEASE", "ALZHEIMERS DISEASE"}:
        return "AD"
    if s in {"MCI", "LMCI", "EMCI", "SMC", "SMCI", "PMCI", "SMCI", "PMCI"}:
        return s
    if "NORMAL" in s or "CONTROL" in s or s == "CN":
        return "CN"
    if "DEMENT" in s or "ALZ" in s or s == "AD":
        return "AD"
    if "MCI" in s:
        return "MCI"
    return s


def standardize_csv(path, modality_name, dataset_name):
    df_raw = pd.read_csv(path).copy()
    df = pd.DataFrame()

    col_image = pick_col(df_raw, ["Image Data ID", "Image ID", "ImageID", "image_id", "Image", "RID Image ID"], required=True)
    col_subj  = pick_col(df_raw, ["Subject", "Subject ID", "PTID", "subject", "subj", "RID"], required=True)
    col_group = pick_col(df_raw, ["Group", "Research Group", "DX", "Diagnosis", "label", "Class"], required=True)
    col_visit = pick_col(df_raw, ["Visit", "Visit Code", "VISCODE", "VISCODE2", "Phase", "visit"], required=False)
    col_date  = pick_col(df_raw, ["Acq Date", "Acquisition Date", "Scan Date", "Study Date", "EXAMDATE", "Date"], required=False)
    col_sex   = pick_col(df_raw, ["Sex", "PTGENDER", "Gender"], required=False)
    col_age   = pick_col(df_raw, ["Age", "AGE", "PTAGE"], required=False)
    col_desc  = pick_col(df_raw, ["Description", "desc", "Sequence", "Protocol"], required=False)

    df["image_id"] = df_raw[col_image].astype(str).str.strip()
    df["subject"] = df_raw[col_subj].astype(str).str.strip()
    df["group"] = df_raw[col_group].map(normalize_group)
    df["visit"] = df_raw[col_visit].astype(str).str.strip() if col_visit else ""
    df["acq_date"] = pd.to_datetime(df_raw[col_date], errors="coerce") if col_date else pd.NaT
    df["sex"] = df_raw[col_sex].astype(str) if col_sex else ""
    df["age"] = pd.to_numeric(df_raw[col_age], errors="coerce") if col_age else np.nan
    df["description"] = df_raw[col_desc].astype(str) if col_desc else ""
    df["dataset"] = dataset_name
    df["modality_name"] = modality_name

    neg = cfg.NEGATIVE_CLASS.upper()
    pos = cfg.POSITIVE_CLASS.upper()
    df = df[df["group"].isin([neg, pos])].copy()
    df["label_num"] = df["group"].map({neg: 0, pos: 1}).astype(int)
    df = df.drop_duplicates(subset=["subject", "image_id", "group", "visit"], keep="first").reset_index(drop=True)
    return df

adni1_mri = standardize_csv(CSV_FILES["ADNI1_MRI"], "MRI", "ADNI1")
adni1_pet = standardize_csv(CSV_FILES["ADNI1_PET"], "PET", "ADNI1")
adni2_mri = standardize_csv(CSV_FILES["ADNI2_MRI"], "MRI", "ADNI2")
adni2_pet = standardize_csv(CSV_FILES["ADNI2_PET"], "PET", "ADNI2")

for name, df in [("ADNI1 MRI", adni1_mri), ("ADNI1 PET", adni1_pet), ("ADNI2 MRI", adni2_mri), ("ADNI2 PET", adni2_pet)]:
    print(f"\n{name}: {df.shape}")
    print(df[["subject", "group", "visit", "image_id", "acq_date"]].head(3))
    print("class counts:", df["group"].value_counts().to_dict())



ADNI1 MRI: (857, 11)
      subject group visit image_id   acq_date
0  137_S_1041    AD    sc   I29268 2006-11-09
1  137_S_1041    AD   m06   I56100 2007-06-06
2  137_S_1041    AD   m12   I84667 2007-12-12
class counts: {'CN': 535, 'AD': 322}

ADNI1 PET: (406, 11)
      subject group visit image_id   acq_date
0  137_S_1041    AD    bl   I32968 2006-12-12
1  137_S_1041    AD   m06   I55363 2007-05-29
2  137_S_1041    AD   m12   I85759 2007-12-18
class counts: {'CN': 232, 'AD': 174}

ADNI2 MRI: (1249, 11)
      subject group visit image_id   acq_date
0  941_S_4376    CN   v02  I269043 2011-06-01
1  941_S_4376    CN   v02  I276860 2012-01-06
2  941_S_4376    CN   v04  I294442 2012-03-30
class counts: {'CN': 905, 'AD': 344}

ADNI2 PET: (600, 11)
      subject group visit image_id   acq_date
0  941_S_4376    CN   v03  I283469 2012-02-08
1  941_S_4365    CN   v03  I274743 2011-12-30
2  941_S_4365    CN   v21  I420334 2014-04-16
class counts: {'CN': 428, 'AD': 172}


In [6]:

def build_nearest_pairs(mri_df, pet_df, dataset_name, max_gap_days=180):
    shared_subjects = sorted(set(mri_df["subject"]) & set(pet_df["subject"]))
    rows = []

    for subj in shared_subjects:
        msub = mri_df[mri_df["subject"] == subj].copy()
        psub = pet_df[pet_df["subject"] == subj].copy()
        best = None
        best_key = None

        for _, mr in msub.iterrows():
            for _, pr in psub.iterrows():
                if mr["group"] != pr["group"]:
                    continue
                if pd.isna(mr["acq_date"]) or pd.isna(pr["acq_date"]):
                    gap_days = 999999
                else:
                    gap_days = abs((mr["acq_date"] - pr["acq_date"]).days)
                same_visit = int(str(mr.get("visit", "")).lower() == str(pr.get("visit", "")).lower())
                score = (gap_days, -same_visit)
                if best is None or score < best_key:
                    best = (mr, pr, gap_days)
                    best_key = score

        if best is None:
            continue
        mr, pr, gap_days = best
        if gap_days != 999999 and gap_days > max_gap_days:
            continue

        rows.append({
            "dataset": dataset_name,
            "subject": subj,
            "group": mr["group"],
            "label_num": int(mr["label_num"]),
            "sex": mr.get("sex", ""),
            "age": mr.get("age", np.nan),
            "mri_visit": mr.get("visit", ""),
            "pet_visit": pr.get("visit", ""),
            "mri_image_id": str(mr["image_id"]),
            "pet_image_id": str(pr["image_id"]),
            "mri_date": mr.get("acq_date", pd.NaT),
            "pet_date": pr.get("acq_date", pd.NaT),
            "pair_gap_days": int(gap_days) if gap_days != 999999 else -1,
        })

    out = pd.DataFrame(rows)
    if len(out) > 0:
        out = out.sort_values(["dataset", "subject"]).reset_index(drop=True)
    return out

paired_adni1 = build_nearest_pairs(adni1_mri, adni1_pet, "ADNI1", max_gap_days=cfg.MAX_PAIR_GAP_DAYS)
paired_adni2 = build_nearest_pairs(adni2_mri, adni2_pet, "ADNI2", max_gap_days=cfg.MAX_PAIR_GAP_DAYS)

if cfg.DEBUG_MAX_PAIRS_PER_DOMAIN is not None:
    # balanced debug sample, if possible
    def debug_sample(df, n):
        if len(df) <= n:
            return df
        parts = []
        per_class = max(1, n // 2)
        for label in sorted(df["label_num"].unique()):
            part = df[df["label_num"] == label].sample(min(per_class, (df["label_num"] == label).sum()), random_state=SEED)
            parts.append(part)
        out = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
        return out
    paired_adni1 = debug_sample(paired_adni1, cfg.DEBUG_MAX_PAIRS_PER_DOMAIN)
    paired_adni2 = debug_sample(paired_adni2, cfg.DEBUG_MAX_PAIRS_PER_DOMAIN)

print("paired_adni1:", paired_adni1.shape, paired_adni1["group"].value_counts().to_dict() if len(paired_adni1) else {})
print("paired_adni2:", paired_adni2.shape, paired_adni2["group"].value_counts().to_dict() if len(paired_adni2) else {})
display(paired_adni1.head())
display(paired_adni2.head())

if len(paired_adni1) == 0 or len(paired_adni2) == 0:
    raise RuntimeError("No paired MRI/PET subjects found for at least one domain. Check CSV labels, subject IDs, and dates.")


paired_adni1: (87, 13) {'CN': 45, 'AD': 42}
paired_adni2: (281, 13) {'CN': 181, 'AD': 100}


,dataset,subject,group,label_num,sex,age,mri_visit,pet_visit,mri_image_id,pet_image_id,mri_date,pet_date,pair_gap_days
0,ADNI1,005_S_0221,AD,1,M,68,m06,m06,I25942,I25918,2006-10-06,2006-10-06,0
1,ADNI1,005_S_0223,CN,0,F,79,m06,m06,I26116,I26108,2006-10-11,2006-10-11,0
2,ADNI1,005_S_0610,CN,0,M,80,m06,m06,I39068,I39087,2007-02-09,2007-02-09,0
3,ADNI1,005_S_0929,AD,1,M,83,m06,m06,I53858,I57389,2007-05-09,2007-06-19,41
4,ADNI1,005_S_1341,AD,1,F,72,m06,m06,I76567,I76770,2007-10-02,2007-10-02,0


,dataset,subject,group,label_num,sex,age,mri_visit,pet_visit,mri_image_id,pet_image_id,mri_date,pet_date,pair_gap_days
0,ADNI2,002_S_0295,CN,0,M,90,v06,v06,I238627,I239487,2011-06-02,2011-06-09,7
1,ADNI2,002_S_0413,CN,0,F,82,v06,v06,I240812,I240813,2011-06-16,2011-06-17,1
2,ADNI2,002_S_0685,CN,0,F,96,v11,v11,I322437,I321228,2012-07-27,2012-08-02,6
3,ADNI2,002_S_1261,CN,0,F,77,v11,v11,I361610,I363184,2013-02-27,2013-03-07,8
4,ADNI2,002_S_1280,CN,0,F,77,v11,v11,I361326,I360640,2013-02-26,2013-02-19,7


In [7]:
# ============================================================
# 5. Build path index only from exact ADNI MRI/PET roots
# ============================================================

VOLUME_EXTENSIONS = (
    ".nii", ".nii.gz", ".img", ".hdr", ".v", ".mnc", ".nrrd", ".mha", ".mhd", ".mgz",
    ".dcm", ".ima"
)

SEARCH_ROOTS = [Path(p) for p in EXTRACTED_ROOTS if Path(p).exists()]

print("Search roots:")
for r in SEARCH_ROOTS:
    print("  ", r)


def build_global_path_index(search_roots):
    index = {"dirs": [], "vol_files": []}
    seen_dirs = set()

    skip_names = {
        ".ipynb_checkpoints",
        "__MACOSX",
        ".git",
        "checkpoint",
        "checkpoints",
        "cache",
    }

    for root in search_roots:
        root = Path(root)
        if not root.exists():
            print(f"[missing] {root}")
            continue

        print(f"\nScanning root: {root}")
        dir_count = 0
        file_count_before = len(index["vol_files"])

        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [d for d in dirnames if d not in skip_names]

            dpath = Path(dirpath)
            low_d = str(dpath).lower()

            if low_d not in seen_dirs:
                index["dirs"].append((low_d, dpath))
                seen_dirs.add(low_d)

            dir_count += 1
            if dir_count % 20000 == 0:
                print(
                    f"  scanned {dir_count:,} dirs | "
                    f"indexed dirs={len(index['dirs']):,}, "
                    f"vol_files={len(index['vol_files']):,}"
                )

            for fn in filenames:
                low_fn = fn.lower()
                if low_fn.endswith(VOLUME_EXTENSIONS):
                    p = dpath / fn
                    low_p = str(p).lower()
                    index["vol_files"].append((low_p, p))

        added_files = len(index["vol_files"]) - file_count_before
        print(
            f"Finished {root}: scanned_dirs={dir_count:,}, "
            f"new_volume_files={added_files:,}, "
            f"total_volume_files={len(index['vol_files']):,}"
        )

    return index


GLOBAL_INDEX = build_global_path_index(SEARCH_ROOTS)

print("\nGLOBAL_INDEX summary:")
print("  dirs:", len(GLOBAL_INDEX["dirs"]))
print("  vol_files:", len(GLOBAL_INDEX["vol_files"]))

if len(GLOBAL_INDEX["vol_files"]) == 0:
    print("\nWARNING: No direct volume files found.")
    print("If your data is DICOM series stored in folders, the notebook may still work using directory paths.")

Search roots:
   /kaggle/input/datasets/syedmuabdullah99313/dl-project/ADNI
   /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni1_pet_clean/ADNI
   /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni2_mri/ADNI
   /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni2_pet_clean/ADNI

Scanning root: /kaggle/input/datasets/syedmuabdullah99313/dl-project/ADNI
Finished /kaggle/input/datasets/syedmuabdullah99313/dl-project/ADNI: scanned_dirs=3,966, new_volume_files=269,880, total_volume_files=269,880

Scanning root: /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni1_pet_clean/ADNI
Finished /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni1_pet_clean/ADNI: scanned_dirs=1,068, new_volume_files=48,500, total_volume_files=318,380

Scanning root: /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni2_mri/ADNI
Finished /kaggle/input/datasets/wajdanrasool/adni-2-pet/improvement2_adni2_mri/ADNI: scanned_dirs=3,361, new_v

In [8]:

# ============================================================
# 6. Locate volume paths
# ============================================================
PAPER_PREPROCESSED_KEYWORDS = (
    "cat12", "spm", "mni", "smwc", "mwp", "warped", "normalized", "normalised",
    "registered", "coreg", "brain", "strip", "skull", "resliced", "resample",
    "smooth", "wc1", "wc2", "mrireg", "petreg"
)


def safe_text(x):
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", str(x))[:180]


def _is_probably_preprocessed_path(path):
    low = str(path).lower()
    return any(k in low for k in PAPER_PREPROCESSED_KEYWORDS)


def modality_terms(modality):
    if modality.upper() == "MRI":
        return ["mri", "mprage", "spgr", "t1", "t1w"]
    return ["pet", "fdg", "av45", "amyloid"]


def _path_under_any_root(path, roots):
    path_s = str(Path(path)).lower().replace("\\", "/")
    for r in roots:
        r_s = str(Path(r)).lower().replace("\\", "/").rstrip("/")
        if path_s == r_s or path_s.startswith(r_s + "/"):
            return True
    return False


def locate_study_path(index, row, modality):
    modality = modality.upper()

    image_id = str(row["mri_image_id"] if modality == "MRI" else row["pet_image_id"]).lower()
    subject = str(row["subject"]).lower()
    visit = str(row["mri_visit"] if modality == "MRI" else row["pet_visit"]).lower()
    dataset = str(row["dataset"]).upper()

    mod_terms = modality_terms(modality)
    allowed_roots = ROOTS_BY_DATASET_MODALITY.get((dataset, modality), [])

    candidates = []

    def add_candidate(p, low, source_type):
        score = 0

        if allowed_roots:
            if _path_under_any_root(p, allowed_roots):
                score += 250
            else:
                score -= 250

        if image_id and image_id not in {"nan", "none", ""} and image_id in low:
            score += 100

        if subject and subject not in {"nan", "none", ""} and subject in low:
            score += 35

        if dataset.lower() in low:
            score += 20

        if any(t in low for t in mod_terms):
            score += 20

        if visit and visit not in {"nan", "none", ""} and visit in low:
            score += 8

        if cfg.PREFER_PREPROCESSED_CANDIDATES and _is_probably_preprocessed_path(p):
            score += 8

        if source_type == "file":
            score += 2

        size = p.stat().st_size if p.exists() and p.is_file() else 0
        candidates.append((score, size, source_type, p))

    for low, p in index["vol_files"]:
        if image_id and image_id not in {"nan", "none", ""} and image_id in low:
            add_candidate(p, low, "file")

    for low, p in index["dirs"]:
        if image_id and image_id not in {"nan", "none", ""} and image_id in low:
            add_candidate(p, low, "dir")

    if not candidates:
        for low, p in index["vol_files"]:
            if subject and subject in low:
                if not allowed_roots or _path_under_any_root(p, allowed_roots):
                    add_candidate(p, low, "file")

        for low, p in index["dirs"]:
            if subject and subject in low:
                if not allowed_roots or _path_under_any_root(p, allowed_roots):
                    add_candidate(p, low, "dir")

    if not candidates:
        for low, p in index["vol_files"]:
            if subject and subject in low:
                add_candidate(p, low, "file")

        for low, p in index["dirs"]:
            if subject and subject in low:
                add_candidate(p, low, "dir")

    if not candidates:
        raise FileNotFoundError(
            f"Could not locate {modality} path for "
            f"dataset={row['dataset']}, subject={row['subject']}, image_id={image_id}"
        )

    candidates = sorted(candidates, key=lambda x: (-x[0], -x[1], len(str(x[3]))))
    best_score, best_size, best_type, best_path = candidates[0]

    if best_score < 0:
        print(
            f"WARNING: weak path match for {dataset} {modality} "
            f"subject={row['subject']} image_id={image_id}: {best_path} score={best_score}"
        )

    return best_path

   
      

In [9]:

# ============================================================
# 6b. Load volumes from NIfTI / Analyze / ECAT / DICOM
# ============================================================
def _ensure_3d_volume(arr, source="volume"):
    arr = np.asarray(arr, dtype=np.float32)
    while arr.ndim > 3:
        arr = arr[..., 0]
    if arr.ndim != 3:
        raise RuntimeError(f"Expected 3D volume from {source}, got shape {arr.shape}")
    return arr.astype(np.float32)


def _load_analyze_like_hdr_pair(hdr_path):
    hdr_path = Path(hdr_path)
    data_candidates = [hdr_path.with_suffix(".img"), Path(str(hdr_path)[:-4])]
    data_path = next((p for p in data_candidates if p.exists()), None)
    if data_path is None:
        raise RuntimeError(f"Missing .img pair for {hdr_path}")
    from nibabel.analyze import AnalyzeHeader
    with open(hdr_path, "rb") as f_hdr:
        hdr = AnalyzeHeader.from_fileobj(f_hdr)
    with open(data_path, "rb") as f_img:
        arr = hdr.data_from_fileobj(f_img)
    return _ensure_3d_volume(arr, f"{hdr_path}+{data_path}")


def load_single_volume_file(path):
    path = Path(path)
    low = str(path).lower()

    if low.endswith(".v"):
        try:
            from nibabel import ecat
            img = ecat.load(str(path))
            return _ensure_3d_volume(img.get_fdata(), path)
        except Exception:
            pass

    if low.endswith(".hdr"):
        try:
            return _load_analyze_like_hdr_pair(path)
        except Exception:
            pass

    try:
        img = nib.load(str(path))
        return _ensure_3d_volume(img.get_fdata(), path)
    except Exception:
        pass

    try:
        img = sitk.ReadImage(str(path))
        arr = sitk.GetArrayFromImage(img).astype(np.float32)
        arr = np.transpose(arr, (2, 1, 0))
        return _ensure_3d_volume(arr, path)
    except Exception as e:
        raise RuntimeError(f"Could not load file {path}") from e


def load_dicom_series_from_dir(series_dir):
    series_dir = Path(series_dir)
    try:
        reader = sitk.ImageSeriesReader()
        series_ids = reader.GetGDCMSeriesIDs(str(series_dir))
        if series_ids:
            candidates = []
            for sid in series_ids:
                files = list(reader.GetGDCMSeriesFileNames(str(series_dir), sid))
                candidates.append((len(files), sid, files))
            for _, sid, files in sorted(candidates, reverse=True):
                try:
                    reader.SetFileNames(files)
                    img = reader.Execute()
                    arr = sitk.GetArrayFromImage(img).astype(np.float32)
                    arr = np.transpose(arr, (2, 1, 0))
                    return _ensure_3d_volume(arr, f"{series_dir}::{sid}")
                except Exception:
                    continue
    except Exception:
        pass

    grouped = {}
    for p in series_dir.rglob("*"):
        if not p.is_file():
            continue
        try:
            ds = pydicom.dcmread(str(p), stop_before_pixels=False, force=True)
            if not hasattr(ds, "pixel_array"):
                continue
            arr = np.asarray(ds.pixel_array, dtype=np.float32)
            if arr.ndim != 2:
                continue
            uid = str(getattr(ds, "SeriesInstanceUID", "unknown"))
            inst = getattr(ds, "InstanceNumber", len(grouped.get(uid, [])))
            try:
                inst = float(inst)
            except Exception:
                inst = len(grouped.get(uid, []))
            grouped.setdefault(uid, []).append((inst, arr))
        except Exception:
            continue

    candidates = []
    for uid, slices in grouped.items():
        if len(slices) < 4:
            continue
        shapes = {s.shape for _, s in slices}
        if len(shapes) != 1:
            continue
        slices = sorted(slices, key=lambda x: x[0])
        vol = np.stack([s for _, s in slices], axis=-1).astype(np.float32)
        candidates.append((vol.size, vol))
    if not candidates:
        raise RuntimeError(f"No readable DICOM series inside {series_dir}")
    return _ensure_3d_volume(sorted(candidates, key=lambda x: x[0], reverse=True)[0][1], series_dir)


def _rank_volume_candidate(p):
    low = str(p).lower()
    size = p.stat().st_size if p.exists() else 0
    pre_bonus = -20 if cfg.PREFER_PREPROCESSED_CANDIDATES and _is_probably_preprocessed_path(p) else 0
    if low.endswith(".nii.gz"): rank = 0
    elif low.endswith(".nii"): rank = 1
    elif low.endswith(".mgz"): rank = 2
    elif low.endswith((".mha", ".mhd")): rank = 3
    elif low.endswith(".img"): rank = 4
    elif low.endswith(".hdr"): rank = 5
    elif low.endswith(".v"): rank = 99
    else: rank = 50
    return (pre_bonus + rank, -size)


def load_volume(path):
    path = Path(path)
    if path.is_file():
        return load_single_volume_file(path)

    # Try as DICOM directory first
    try:
        return load_dicom_series_from_dir(path)
    except Exception:
        pass

    # Otherwise search volume files inside directory
    volume_files = []
    for p in path.rglob("*"):
        if p.is_file() and str(p).lower().endswith(VOLUME_EXTENSIONS):
            volume_files.append(p)
    if not volume_files:
        raise RuntimeError(f"No volume files found inside {path}")

    last_error = None
    for vf in sorted(volume_files, key=_rank_volume_candidate):
        try:
            return load_single_volume_file(vf)
        except Exception as e:
            last_error = e
    raise RuntimeError(f"All loaders failed for {path}") from last_error


In [10]:

# ============================================================
# 6c. Preprocess and cache
# ============================================================
def _ensure_3d_float(volume):
    v = np.asarray(volume, dtype=np.float32)
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
    while v.ndim > 3:
        v = v[..., 0]
    if v.ndim != 3:
        raise RuntimeError(f"Expected 3D, got {v.shape}")
    return v.astype(np.float32)


def robust_clip(volume, lower_q=0.5, upper_q=99.5):
    v = _ensure_3d_float(volume)
    finite = np.isfinite(v)
    if finite.sum() == 0:
        return np.zeros_like(v, dtype=np.float32)
    lo, hi = np.percentile(v[finite], [lower_q, upper_q])
    return np.clip(v, lo, hi).astype(np.float32)


def crop_to_foreground(volume, modality, margin=4):
    v = _ensure_3d_float(volume)
    nonzero = np.abs(v) > 0
    fg_vals = v[nonzero] if nonzero.any() else v[np.isfinite(v)]
    if fg_vals.size == 0:
        return v
    threshold = np.percentile(fg_vals, 35 if modality.upper() == "MRI" else 55)
    mask = (v > threshold) | nonzero
    coords = np.argwhere(mask)
    if coords.size == 0:
        return v
    lo = np.maximum(coords.min(axis=0) - margin, 0)
    hi = np.minimum(coords.max(axis=0) + margin + 1, np.array(v.shape))
    cropped = v[int(lo[0]):int(hi[0]), int(lo[1]):int(hi[1]), int(lo[2]):int(hi[2])]
    return cropped.astype(np.float32) if cropped.size > 0 else v


def maybe_gaussian_smooth(volume, sigma):
    v = _ensure_3d_float(volume)
    if sigma is None or sigma <= 0:
        return v
    try:
        itk = sitk.GetImageFromArray(np.transpose(v, (2, 1, 0)))
        smoothed = sitk.SmoothingRecursiveGaussian(itk, sigma=float(sigma))
        out = sitk.GetArrayFromImage(smoothed).astype(np.float32)
        return np.transpose(out, (2, 1, 0))
    except Exception:
        return v


def normalize_mri(volume):
    v = robust_clip(volume)
    mask = v > np.percentile(v, 20)
    vals = v[mask] if mask.sum() > 0 else v.reshape(-1)
    return ((v - vals.mean()) / (vals.std() + 1e-6)).astype(np.float32)


def normalize_pet(volume):
    v = robust_clip(volume)
    v = v - v.min()
    return (v / (v.max() + 1e-6)).astype(np.float32)


def resize_volume(volume, target_shape):
    x = torch.tensor(volume, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    x = F.interpolate(x, size=tuple(target_shape), mode="trilinear", align_corners=False)
    return x.squeeze(0).squeeze(0).cpu().numpy().astype(np.float32)


def preprocess_volume(volume, modality, source_path=None):
    v = _ensure_3d_float(volume)
    source_path = None if source_path is None else Path(source_path)
    paper_like = source_path is not None and cfg.PREFER_PREPROCESSED_CANDIDATES and _is_probably_preprocessed_path(source_path)

    if cfg.RAW_APPROX_FOREGROUND_CROP and not paper_like:
        v = crop_to_foreground(v, modality, margin=4)
    if not paper_like:
        v = maybe_gaussian_smooth(v, sigma=cfg.RAW_APPROX_GAUSSIAN_SIGMA)

    if modality.upper() == "MRI":
        v = normalize_mri(v)
    else:
        v = normalize_pet(v)

    if tuple(v.shape) != tuple(cfg.TARGET_SHAPE):
        v = resize_volume(v, cfg.TARGET_SHAPE)

    # normalize again after interpolation
    if modality.upper() == "MRI":
        v = normalize_mri(v)
    else:
        v = normalize_pet(v)
    return v.astype(np.float32)


def cache_key(row, modality):
    image_id = row["mri_image_id"] if modality.upper() == "MRI" else row["pet_image_id"]
    return safe_text(f"{row['dataset']}__{row['subject']}__{modality}__{image_id}__{cfg.TARGET_SHAPE}.npy")


def get_cached_or_build(row, modality):
    out_path = CACHE_ROOT / cache_key(row, modality)
    if out_path.exists():
        try:
            return np.load(out_path).astype(np.float32)
        except Exception:
            try: out_path.unlink()
            except Exception: pass

    source_path = locate_study_path(GLOBAL_INDEX, row, modality)
    vol = load_volume(source_path)
    arr = preprocess_volume(vol, modality=modality, source_path=source_path)
    np.save(out_path, arr.astype(np.float32))
    return arr.astype(np.float32)


In [11]:


try:
    import signal
    SIGNAL_AVAILABLE = hasattr(signal, "SIGALRM")
except Exception:
    SIGNAL_AVAILABLE = False

class TimeoutException(Exception):
    pass

def _timeout_handler(signum, frame):
    raise TimeoutException("Image loading timed out")

if SIGNAL_AVAILABLE:
    signal.signal(signal.SIGALRM, _timeout_handler)


def filter_loadable_pairs(df, name, timeout_sec=90):
    df = df.reset_index(drop=True).copy()
    kept, bad = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"filter {name}"):
        try:
            if SIGNAL_AVAILABLE:
                signal.alarm(timeout_sec)
            if cfg.INPUT_MODE in ["MULTIMODAL", "MRI_ONLY"]:
                _ = get_cached_or_build(row, "MRI")
            if SIGNAL_AVAILABLE:
                signal.alarm(timeout_sec)
            if cfg.INPUT_MODE in ["MULTIMODAL", "PET_ONLY"]:
                _ = get_cached_or_build(row, "PET")
            if SIGNAL_AVAILABLE:
                signal.alarm(0)
            kept.append(row)
        except Exception as e:
            if SIGNAL_AVAILABLE:
                signal.alarm(0)
            bad_row = row.copy()
            bad_row["drop_reason"] = str(e)[:500]
            bad.append(bad_row)
    kept_df = pd.DataFrame(kept).reset_index(drop=True)
    bad_df = pd.DataFrame(bad).reset_index(drop=True)
    print(f"{name}: kept {len(kept_df)} / {len(df)} | dropped {len(bad_df)}")
    return kept_df, bad_df


def safe_train_test_split(df, test_size, seed, stratify_col="label_num"):
    strat = df[stratify_col] if df[stratify_col].value_counts().min() >= 2 else None
    return train_test_split(df, test_size=test_size, random_state=seed, stratify=strat)


def make_domain_splits(df, seed=SEED):
    train_val, test = safe_train_test_split(df, cfg.TEST_SIZE, seed)
    train, val = safe_train_test_split(train_val, cfg.VAL_FROM_TRAIN_RATIO, seed)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

if cfg.PREFILTER_BEFORE_SPLIT:
    paired_adni1_f, bad_adni1 = filter_loadable_pairs(paired_adni1, "ADNI1", cfg.LOAD_TIMEOUT_SEC)
    paired_adni2_f, bad_adni2 = filter_loadable_pairs(paired_adni2, "ADNI2", cfg.LOAD_TIMEOUT_SEC)
else:
    paired_adni1_f, bad_adni1 = paired_adni1.copy(), pd.DataFrame()
    paired_adni2_f, bad_adni2 = paired_adni2.copy(), pd.DataFrame()

if len(paired_adni1_f) < 4 or len(paired_adni2_f) < 4:
    raise RuntimeError("Too few loadable pairs after filtering. Check paths/data extraction/indexing.")

adni1_train, adni1_val, adni1_test = make_domain_splits(paired_adni1_f)
adni2_train, adni2_val, adni2_test = make_domain_splits(paired_adni2_f)

print("\nADNI1 usable/splits:", len(paired_adni1_f), len(adni1_train), len(adni1_val), len(adni1_test))
for split_name, dfx in [("train", adni1_train), ("val", adni1_val), ("test", adni1_test)]:
    print("ADNI1", split_name, dfx["group"].value_counts().to_dict())
print("\nADNI2 usable/splits:", len(paired_adni2_f), len(adni2_train), len(adni2_val), len(adni2_test))
for split_name, dfx in [("train", adni2_train), ("val", adni2_val), ("test", adni2_test)]:
    print("ADNI2", split_name, dfx["group"].value_counts().to_dict())

if len(bad_adni1) + len(bad_adni2) > 0:
    dropped_path = WORK_ROOT / "dropped_unreadable_pairs.csv"
    pd.concat([bad_adni1, bad_adni2], ignore_index=True).to_csv(dropped_path, index=False)
    print("Dropped-pair log saved to:", dropped_path)


filter ADNI1:   0%|          | 0/87 [00:00<?, ?it/s]

sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix


sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix


sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24948 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix


ADNI1: kept 72 / 87 | dropped 15


filter ADNI2:   0%|          | 0/281 [00:00<?, ?it/s]

sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr

sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix


sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix
sizeof_hdr should be 348; set sizeof_hdr to 348
data code 24951 not recognized; not attempting fix


ADNI2: kept 260 / 281 | dropped 21

ADNI1 usable/splits: 72 28 22 22
ADNI1 train {'CN': 15, 'AD': 13}
ADNI1 val {'CN': 11, 'AD': 11}
ADNI1 test {'CN': 11, 'AD': 11}

ADNI2 usable/splits: 260 104 78 78
ADNI2 train {'CN': 67, 'AD': 37}
ADNI2 val {'CN': 50, 'AD': 28}
ADNI2 test {'CN': 50, 'AD': 28}
Dropped-pair log saved to: /kaggle/working/daan_adni_work/dropped_unreadable_pairs.csv


In [12]:


class PairedADNIDataset(Dataset):
    def __init__(self, df, split_name="split"):
        self.df = df.reset_index(drop=True).copy()
        self.split_name = split_name

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx].to_dict()

        if cfg.INPUT_MODE in ["MULTIMODAL", "MRI_ONLY"]:
            mri = torch.tensor(get_cached_or_build(row, "MRI"), dtype=torch.float32).unsqueeze(0)
        else:
            mri = torch.zeros((1, *cfg.TARGET_SHAPE), dtype=torch.float32)

        if cfg.INPUT_MODE in ["MULTIMODAL", "PET_ONLY"]:
            pet = torch.tensor(get_cached_or_build(row, "PET"), dtype=torch.float32).unsqueeze(0)
        else:
            pet = torch.zeros((1, *cfg.TARGET_SHAPE), dtype=torch.float32)

        label = torch.tensor(int(row["label_num"]), dtype=torch.long)
        return {
            "mri": mri,
            "pet": pet,
            "label": label,
            "subject": str(row["subject"]),
            "dataset": str(row["dataset"]),
        }


def make_loader(df, split_name, shuffle):
    ds = PairedADNIDataset(df, split_name=split_name)
    return DataLoader(
        ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=shuffle,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

sample = adni1_train.iloc[0].to_dict()
if cfg.INPUT_MODE in ["MULTIMODAL", "MRI_ONLY"]:
    print("MRI sample shape:", get_cached_or_build(sample, "MRI").shape)
if cfg.INPUT_MODE in ["MULTIMODAL", "PET_ONLY"]:
    print("PET sample shape:", get_cached_or_build(sample, "PET").shape)


MRI sample shape: (96, 112, 96)
PET sample shape: (96, 112, 96)


In [13]:


class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


def grad_reverse(x, lambd=1.0):
    return GradReverse.apply(x, lambd)


class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, padding=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, stride=1, padding=padding, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x):
        return self.net(x)


class Encoder3D(nn.Module):
    def __init__(self, out_channels=128):
        super().__init__()
        channels = [8, 8, 16, 16, 32, 32, 64, 64, out_channels]
        blocks = []
        in_ch = 1
        for out_ch in channels:
            blocks.append(ConvBlock3D(in_ch, out_ch, padding=1))
            in_ch = out_ch
        self.blocks = nn.ModuleList(blocks)
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.output_dim = out_channels

    def forward(self, x):
        for i, block in enumerate(self.blocks, start=1):
            x = block(x)
            if i in {2, 4, 6, 8}:
                x = self.pool(x)
        x = self.gap(x).flatten(1)
        return x


class Discriminator(nn.Module):
    def __init__(self, in_dim=256, hidden_dim=256, num_domains=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_domains),
        )
    def forward(self, x):
        return self.net(x)


class DAAN3D(nn.Module):
    def __init__(self, num_classes=2, feature_dim=256, input_mode="MULTIMODAL"):
        super().__init__()
        self.num_classes = num_classes
        self.feature_dim = feature_dim
        self.input_mode = input_mode.upper()

        self.mri_encoder = Encoder3D(out_channels=128)
        self.pet_encoder = Encoder3D(out_channels=128)

        if self.input_mode == "MULTIMODAL":
            fusion_in = 128 * 3  # MRI, PET, abs diff
        else:
            fusion_in = 128

        self.bottleneck = nn.Sequential(
            nn.Linear(fusion_in, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

        self.global_discriminator = Discriminator(in_dim=feature_dim, hidden_dim=256, num_domains=2)
        self.local_discriminators = nn.ModuleList([
            Discriminator(in_dim=feature_dim, hidden_dim=256, num_domains=2)
            for _ in range(num_classes)
        ])

    def encode(self, mri, pet):
        if self.input_mode == "MULTIMODAL":
            m = self.mri_encoder(mri)
            p = self.pet_encoder(pet)
            z = torch.cat([m, p, torch.abs(m - p)], dim=1)
        elif self.input_mode == "MRI_ONLY":
            m = self.mri_encoder(mri)
            z = m
        elif self.input_mode == "PET_ONLY":
            p = self.pet_encoder(pet)
            z = p
        else:
            raise ValueError(f"Unknown input_mode: {self.input_mode}")
        feat = self.bottleneck(z)
        return feat

    def classify_from_features(self, feat):
        return self.classifier(feat)

    def forward(self, mri, pet):
        feat = self.encode(mri, pet)
        logits = self.classify_from_features(feat)
        return feat, logits

    def global_domain_logits(self, feat, lambd):
        return self.global_discriminator(grad_reverse(feat, lambd))

    def local_domain_logits(self, feat, class_probs, lambd):
        rev_feat = grad_reverse(feat, lambd)
        outs = []
        for c in range(self.num_classes):
            weight = class_probs[:, c].unsqueeze(1)
            outs.append(self.local_discriminators[c](weight * rev_feat))
        return outs


In [14]:

# ============================================================
# 10. Losses and metrics
# ============================================================
def grl_lambda(progress, gamma=10.0):
    # DANN/DAAN-style schedule: starts near 0, approaches 1
    progress = min(max(float(progress), 0.0), 1.0)
    return float(2.0 / (1.0 + math.exp(-gamma * progress)) - 1.0)


class WeightedFocalLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=2.0):
        super().__init__()
        if class_weights is None:
            self.register_buffer("class_weights", torch.ones(2))
        else:
            self.register_buffer("class_weights", torch.tensor(class_weights, dtype=torch.float32))
        self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.class_weights.to(logits.device), reduction="none")
        pt = torch.exp(-ce)
        return (((1.0 - pt) ** self.gamma) * ce).mean()


def compute_class_weights(df):
    counts = df["label_num"].value_counts().to_dict()
    total = sum(counts.values())
    weights = []
    for c in range(cfg.NUM_CLASSES):
        weights.append(total / max(cfg.NUM_CLASSES * counts.get(c, 0), 1))
    return weights


def make_classification_loss(source_train_df):
    weights = compute_class_weights(source_train_df)
    print("Class weights [negative, positive]:", weights)
    if cfg.CLASS_LOSS.lower() == "focal":
        return WeightedFocalLoss(weights, gamma=cfg.FOCAL_GAMMA).to(DEVICE)
    return nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))


class DAANLossController:
    def __init__(self, dynamic_factor_init=0.5, use_dynamic=True):
        self.dynamic_factor = float(dynamic_factor_init)
        self.use_dynamic = bool(use_dynamic)
        self.global_losses = []
        self.local_losses = []

    def compute(self, model, src_feat, tgt_feat, src_logits, tgt_logits, lambd):
        domain_ce = nn.CrossEntropyLoss()
        bsz_s = src_feat.size(0)
        bsz_t = tgt_feat.size(0)
        src_domain = torch.ones(bsz_s, dtype=torch.long, device=src_feat.device)   
        tgt_domain = torch.zeros(bsz_t, dtype=torch.long, device=tgt_feat.device) 

        src_g = model.global_domain_logits(src_feat, lambd)
        tgt_g = model.global_domain_logits(tgt_feat, lambd)
        global_loss = 0.5 * (domain_ce(src_g, src_domain) + domain_ce(tgt_g, tgt_domain))

        src_probs = F.softmax(src_logits.detach(), dim=1)
        tgt_probs = F.softmax(tgt_logits.detach(), dim=1)
        src_l_list = model.local_domain_logits(src_feat, src_probs, lambd)
        tgt_l_list = model.local_domain_logits(tgt_feat, tgt_probs, lambd)
        local_loss = 0.0
        for c in range(model.num_classes):
            local_loss = local_loss + 0.5 * (domain_ce(src_l_list[c], src_domain) + domain_ce(tgt_l_list[c], tgt_domain))
        local_loss = local_loss / model.num_classes

        self.global_losses.append(float(global_loss.detach().cpu().item()))
        self.local_losses.append(float(local_loss.detach().cpu().item()))

        df = self.dynamic_factor
        adv = (1.0 - df) * cfg.GLOBAL_LOSS_WEIGHT * global_loss + df * cfg.LOCAL_LOSS_WEIGHT * local_loss
        return adv, global_loss.detach(), local_loss.detach(), df

    def update_dynamic_factor(self):
        if not self.use_dynamic or len(self.global_losses) == 0:
            self.global_losses.clear(); self.local_losses.clear()
            return self.dynamic_factor
        g = float(np.mean(self.global_losses))
        l = float(np.mean(self.local_losses))
        new_factor = l / (g + l + 1e-8)
        self.dynamic_factor = float(np.clip(new_factor, 0.10, 0.90))
        self.global_losses.clear(); self.local_losses.clear()
        return self.dynamic_factor


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred) if len(y_true) else np.nan
    f1 = f1_score(y_true, y_pred, zero_division=0) if len(y_true) else np.nan
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    sen = tp / (tp + fn + 1e-8)
    spe = tn / (tn + fp + 1e-8)
    bal_acc = 0.5 * (sen + spe)
    return {
        "ACC": acc * 100,
        "BAL_ACC": bal_acc * 100,
        "SEN": sen * 100,
        "SPE": spe * 100,
        "AUC": auc * 100 if not np.isnan(auc) else np.nan,
        "F1": f1 * 100,
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
        "threshold": float(threshold),
    }


def validation_score(metrics):
    key = cfg.CHECKPOINT_MONITOR.upper()
    value = metrics.get(key, np.nan)
    if not np.isnan(value):
        return float(value)
    for k in ["BAL_ACC", "AUC", "ACC", "F1"]:
        value = metrics.get(k, np.nan)
        if not np.isnan(value):
            return float(value)
    return float("-inf")


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    y_true, y_prob, subjects, datasets = [], [], [], []
    for batch in loader:
        mri = batch["mri"].to(DEVICE, non_blocking=True)
        pet = batch["pet"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        feat, logits = model(mri, pet)
        prob = F.softmax(logits, dim=1)[:, 1]
        y_true.extend(label.cpu().numpy().tolist())
        y_prob.extend(prob.detach().cpu().numpy().tolist())
        subjects.extend(batch["subject"])
        datasets.extend(batch["dataset"])
    return np.asarray(y_true), np.asarray(y_prob), subjects, datasets


def evaluate_model(model, loader, threshold=0.5):
    y, p, _, _ = collect_predictions(model, loader)
    return compute_metrics(y, p, threshold=threshold)


def print_metrics(prefix, metrics):
    keys = ["ACC", "BAL_ACC", "SEN", "SPE", "AUC", "F1", "TP", "TN", "FP", "FN"]
    msg = prefix + " | " + " | ".join([f"{k}={metrics[k]:.2f}" if isinstance(metrics[k], float) else f"{k}={metrics[k]}" for k in keys])
    print(msg)


In [15]:

# ============================================================
# 11. Training functions
# ============================================================
def train_pretrain_epoch(model, loader, optimizer, scaler, clf_loss_fn):
    model.train()
    total_loss, n = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(tqdm(loader, leave=False, desc="pretrain"), start=1):
        mri = batch["mri"].to(DEVICE, non_blocking=True)
        pet = batch["pet"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(cfg.AMP and DEVICE.type == "cuda")):
            feat, logits = model(mri, pet)
            loss = clf_loss_fn(logits, label)
            loss = loss / cfg.GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()
        if step % cfg.GRAD_ACCUM_STEPS == 0 or step == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        total_loss += float(loss.detach().cpu().item()) * cfg.GRAD_ACCUM_STEPS
        n += 1
    return total_loss / max(n, 1)


def train_adapt_epoch(model, source_loader, target_loader, optimizer, scaler, clf_loss_fn, daan_ctrl, epoch_idx, total_epochs):
    model.train()
    total, total_cls, total_adv, total_g, total_l = 0.0, 0.0, 0.0, 0.0, 0.0
    n = 0
    steps = max(len(source_loader), len(target_loader))
    src_iter = cycle(source_loader)
    tgt_iter = cycle(target_loader)
    optimizer.zero_grad(set_to_none=True)

    for step in tqdm(range(1, steps + 1), leave=False, desc="adapt"):
        s_batch = next(src_iter)
        t_batch = next(tgt_iter)
        smri = s_batch["mri"].to(DEVICE, non_blocking=True)
        spet = s_batch["pet"].to(DEVICE, non_blocking=True)
        sy = s_batch["label"].to(DEVICE, non_blocking=True)
        tmri = t_batch["mri"].to(DEVICE, non_blocking=True)
        tpet = t_batch["pet"].to(DEVICE, non_blocking=True)

        global_step = (epoch_idx - 1) * steps + step
        progress = global_step / max(total_epochs * steps, 1)
        lambd = grl_lambda(progress, gamma=cfg.DAAN_GAMMA)

        with torch.cuda.amp.autocast(enabled=(cfg.AMP and DEVICE.type == "cuda")):
            s_feat, s_logits = model(smri, spet)
            t_feat, t_logits = model(tmri, tpet)
            cls_loss = clf_loss_fn(s_logits, sy)
            adv_loss, g_loss, l_loss, dyn_factor = daan_ctrl.compute(model, s_feat, t_feat, s_logits, t_logits, lambd)
            loss = cls_loss + cfg.ADV_WEIGHT * adv_loss

            if cfg.MODALITY_ALIGN_WEIGHT > 0 and cfg.INPUT_MODE == "MULTIMODAL":
                # optional: weakly stabilize MRI/PET feature space; not part of pure DAAN
                # disabled by default
                pass

            loss = loss / cfg.GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        if step % cfg.GRAD_ACCUM_STEPS == 0 or step == steps:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total += float(loss.detach().cpu().item()) * cfg.GRAD_ACCUM_STEPS
        total_cls += float(cls_loss.detach().cpu().item())
        total_adv += float(adv_loss.detach().cpu().item())
        total_g += float(g_loss.detach().cpu().item())
        total_l += float(l_loss.detach().cpu().item())
        n += 1

    new_dyn = daan_ctrl.update_dynamic_factor()
    return {
        "loss": total / max(n, 1),
        "cls": total_cls / max(n, 1),
        "adv": total_adv / max(n, 1),
        "global": total_g / max(n, 1),
        "local": total_l / max(n, 1),
        "dynamic_factor": new_dyn,
    }


def run_direction(source_name, target_name, source_train_df, source_val_df, source_test_df, target_train_df, target_val_df, target_test_df):
    print("=" * 100)
    print(f"Running DAAN: {source_name} -> {target_name} | input_mode={cfg.INPUT_MODE}")
    print("=" * 100)
    print(f"source train/val/test = {len(source_train_df)}/{len(source_val_df)}/{len(source_test_df)}")
    print(f"target train/val/test = {len(target_train_df)}/{len(target_val_df)}/{len(target_test_df)}")

    source_train_loader = make_loader(source_train_df, f"{source_name}_train", shuffle=True)
    source_val_loader   = make_loader(source_val_df,   f"{source_name}_val",   shuffle=False)
    source_test_loader  = make_loader(source_test_df,  f"{source_name}_test",  shuffle=False)
    target_train_loader = make_loader(target_train_df, f"{target_name}_train_unlabeled", shuffle=True)
    target_val_loader   = make_loader(target_val_df,   f"{target_name}_val_report", shuffle=False)
    target_test_loader  = make_loader(target_test_df,  f"{target_name}_test_report", shuffle=False)

    model = DAAN3D(num_classes=cfg.NUM_CLASSES, feature_dim=cfg.FEATURE_DIM, input_mode=cfg.INPUT_MODE).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.AMP and DEVICE.type == "cuda"))
    clf_loss_fn = make_classification_loss(source_train_df)
    daan_ctrl = DAANLossController(cfg.DYNAMIC_FACTOR_INIT, cfg.USE_DYNAMIC_FACTOR)

    exp_key = f"DAAN_{cfg.INPUT_MODE}_{source_name}_to_{target_name}"
    best_ckpt = CHECKPOINT_ROOT / f"{exp_key}_best.pt"
    history = []
    best_score = float("-inf")
    patience = 0

    # Optional source-only warmup
    for epoch in range(1, cfg.PRETRAIN_EPOCHS + 1):
        train_loss = train_pretrain_epoch(model, source_train_loader, optimizer, scaler, clf_loss_fn)
        val_m = evaluate_model(model, source_val_loader)
        score = validation_score(val_m)
        history.append({"stage": "pretrain", "epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_m.items()}})
        if score > best_score:
            best_score = score
            patience = 0
            torch.save({"model": model.state_dict(), "cfg": asdict(cfg), "best_score": best_score}, best_ckpt)
        else:
            patience += 1
        print(f"[pretrain {epoch:03d}/{cfg.PRETRAIN_EPOCHS}] loss={train_loss:.4f} | val_score={score:.2f} | best={best_score:.2f}")
        print_metrics("  source-val", val_m)

    # DAAN adaptation
    for epoch in range(1, cfg.ADAPT_EPOCHS + 1):
        train_stats = train_adapt_epoch(model, source_train_loader, target_train_loader, optimizer, scaler, clf_loss_fn, daan_ctrl, epoch, cfg.ADAPT_EPOCHS)
        src_val_m = evaluate_model(model, source_val_loader)
        tgt_test_m = evaluate_model(model, target_test_loader)  # reported only, not used for checkpointing
        score = validation_score(src_val_m)

        row = {"stage": "adapt", "epoch": epoch, **{f"train_{k}": v for k, v in train_stats.items()}, **{f"src_val_{k}": v for k, v in src_val_m.items()}, **{f"tgt_test_{k}": v for k, v in tgt_test_m.items()}}
        history.append(row)

        if score > best_score:
            best_score = score
            patience = 0
            torch.save({"model": model.state_dict(), "cfg": asdict(cfg), "best_score": best_score}, best_ckpt)
        else:
            patience += 1

        print(
            f"[adapt {epoch:03d}/{cfg.ADAPT_EPOCHS}] "
            f"loss={train_stats['loss']:.4f} cls={train_stats['cls']:.4f} adv={train_stats['adv']:.4f} "
            f"g={train_stats['global']:.4f} l={train_stats['local']:.4f} dyn={train_stats['dynamic_factor']:.3f} | "
            f"src_val_score={score:.2f} best={best_score:.2f} patience={patience}"
        )
        print_metrics("  source-val", src_val_m)
        print_metrics("  target-test(report only)", tgt_test_m)

        if patience >= cfg.EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at DAAN epoch {epoch}")
            break

    # Load best source-val checkpoint
    ckpt = torch.load(best_ckpt, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])

    final = {
        "source_val": evaluate_model(model, source_val_loader),
        "source_test": evaluate_model(model, source_test_loader),
        "target_val_report": evaluate_model(model, target_val_loader),
        "target_test_report": evaluate_model(model, target_test_loader),
    }

    for name, m in final.items():
        print_metrics(f"FINAL {name}", m)

    # Save outputs
    hist_df = pd.DataFrame(history)
    hist_path = WORK_ROOT / f"{exp_key}_history.csv"
    hist_df.to_csv(hist_path, index=False)

    y, p, subjects, datasets = collect_predictions(model, target_test_loader)
    pred_df = pd.DataFrame({
        "dataset": datasets,
        "subject": subjects,
        "label": y,
        "prob_positive": p,
        "pred": (p >= 0.5).astype(int),
    })
    pred_path = WORK_ROOT / f"{exp_key}_target_test_predictions.csv"
    pred_df.to_csv(pred_path, index=False)

    summary = {
        "experiment": exp_key,
        "source": source_name,
        "target": target_name,
        "input_mode": cfg.INPUT_MODE,
        "best_source_val_score": float(ckpt["best_score"]),
        "checkpoint": str(best_ckpt),
        "history_csv": str(hist_path),
        "target_test_predictions_csv": str(pred_path),
        "n_source_train": int(len(source_train_df)),
        "n_source_val": int(len(source_val_df)),
        "n_source_test": int(len(source_test_df)),
        "n_target_train_unlabeled": int(len(target_train_df)),
        "n_target_val_report": int(len(target_val_df)),
        "n_target_test_report": int(len(target_test_df)),
        "metrics": final,
    }
    summary_path = WORK_ROOT / f"{exp_key}_summary.json"
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)
    print("Saved summary:", summary_path)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


In [16]:


directions = []
if cfg.RUN_BOTH_DIRECTIONS:
    directions = [
        ("ADNI1", "ADNI2", adni1_train, adni1_val, adni1_test, adni2_train, adni2_val, adni2_test),
        ("ADNI2", "ADNI1", adni2_train, adni2_val, adni2_test, adni1_train, adni1_val, adni1_test),
    ]
else:
    if cfg.RUN_DIRECTION == "ADNI1_TO_ADNI2":
        directions = [("ADNI1", "ADNI2", adni1_train, adni1_val, adni1_test, adni2_train, adni2_val, adni2_test)]
    elif cfg.RUN_DIRECTION == "ADNI2_TO_ADNI1":
        directions = [("ADNI2", "ADNI1", adni2_train, adni2_val, adni2_test, adni1_train, adni1_val, adni1_test)]
    else:
        raise ValueError(f"Unknown RUN_DIRECTION: {cfg.RUN_DIRECTION}")

all_summaries = {}
for args in directions:
    summary = run_direction(*args)
    all_summaries[summary["experiment"]] = summary

all_summary_path = WORK_ROOT / f"DAAN_{cfg.INPUT_MODE}_all_results.json"
with open(all_summary_path, "w") as f:
    json.dump(all_summaries, f, indent=2)
print("\nAll summaries saved to:", all_summary_path)


Running DAAN: ADNI1 -> ADNI2 | input_mode=MULTIMODAL
source train/val/test = 28/22/22
target train/val/test = 104/78/78
Class weights [negative, positive]: [0.9333333333333333, 1.0769230769230769]


pretrain:   0%|          | 0/28 [00:00<?, ?it/s]

[pretrain 001/5] loss=0.6951 | val_score=45.45 | best=45.45
  source-val | ACC=45.45 | BAL_ACC=45.45 | SEN=81.82 | SPE=9.09 | AUC=39.67 | F1=60.00 | TP=9 | TN=1 | FP=10 | FN=2


pretrain:   0%|          | 0/28 [00:00<?, ?it/s]

[pretrain 002/5] loss=0.6721 | val_score=40.91 | best=45.45
  source-val | ACC=40.91 | BAL_ACC=40.91 | SEN=18.18 | SPE=63.64 | AUC=32.23 | F1=23.53 | TP=2 | TN=7 | FP=4 | FN=9


...

[adapt 020/25] loss=0.8033 cls=0.1058 adv=0.6975 g=0.7080 l=0.6864 dyn=0.492 | src_val_score=49.43 best=61.71 patience=6
  source-val | ACC=37.18 | BAL_ACC=49.43 | SEN=92.86 | SPE=6.00 | AUC=50.86 | F1=51.49 | TP=26 | TN=3 | FP=47 | FN=2
  target-test(report only) | ACC=50.00 | BAL_ACC=50.00 | SEN=100.00 | SPE=0.00 | AUC=60.33 | F1=66.67 | TP=11 | TN=0 | FP=11 | FN=0


adapt:   0%|          | 0/104 [00:00<?, ?it/s]

[adapt 021/25] loss=0.7091 cls=0.0129 adv=0.6962 g=0.7100 l=0.6819 dyn=0.490 | src_val_score=47.29 best=61.71 patience=7
  source-val | ACC=38.46 | BAL_ACC=47.29 | SEN=78.57 | SPE=16.00 | AUC=49.79 | F1=47.83 | TP=22 | TN=8 | FP=42 | FN=6
  target-test(report only) | ACC=50.00 | BAL_ACC=50.00 | SEN=63.64 | SPE=36.36 | AUC=52.07 | F1=56.00 | TP=7 | TN=4 | FP=7 | FN=4


adapt:   0%|          | 0/104 [00:00<?, ?it/s]

[adapt 022/25] loss=0.7063 cls=0.0088 adv=0.6976 g=0.7027 l=0.6923 dyn=0.496 | src_val_score=45.50 best=61.71 patience=8
  source-val | ACC=37.18 | BAL_ACC=45.50 | SEN=75.00 | SPE=16.00 | AUC=54.00 | F1=46.15 | TP=21 | TN=8 | FP=42 | FN=7
  target-test(report only) | ACC=45.45 | BAL_ACC=45.45 | SEN=72.73 | SPE=18.18 | AUC=56.20 | F1=57.14 | TP=8 | TN=2 | FP=9 | FN=3
Early stopping at DAAN epoch 22
FINAL source_val | ACC=58.97 | BAL_ACC=61.71 | SEN=71.43 | SPE=52.00 | AUC=64.29 | F1=55.56 | TP=20 | TN=26 | FP=24 | FN=8
FINAL source_test | ACC=46.15 | BAL_ACC=51.71 | SEN=71.43 | SPE=32.00 | AUC=51.86 | F1=48.78 | TP=20 | TN=16 | FP=34 | FN=8
FINAL target_val_report | ACC=50.00 | BAL_ACC=50.00 | SEN=45.45 | SPE=54.55 | AUC=55.37 | F1=47.62 | TP=5 | TN=6 | FP=5 | FN=6
FINAL target_test_report | ACC=59.09 | BAL_ACC=59.09 | SEN=45.45 | SPE=72.73 | AUC=60.33 | F1=52.63 | TP=5 | TN=8 | FP=3 | FN=6
Saved summary: /kaggle/working/daan_adni_work/DAAN_MULTIMODAL_ADNI2_to_ADNI1_summary.json

All su

In [17]:


rows = []
for exp, s in all_summaries.items():
    for split_name, metrics in s["metrics"].items():
        row = {
            "experiment": exp,
            "split": split_name,
            "ACC": metrics["ACC"],
            "BAL_ACC": metrics["BAL_ACC"],
            "SEN": metrics["SEN"],
            "SPE": metrics["SPE"],
            "AUC": metrics["AUC"],
            "F1": metrics["F1"],
            "TP": metrics["TP"],
            "TN": metrics["TN"],
            "FP": metrics["FP"],
            "FN": metrics["FN"],
        }
        rows.append(row)

result_table = pd.DataFrame(rows)
result_csv = WORK_ROOT / f"DAAN_{cfg.INPUT_MODE}_compact_results.csv"
result_table.to_csv(result_csv, index=False)
print("Saved compact result table to:", result_csv)
display(result_table)


Saved compact result table to: /kaggle/working/daan_adni_work/DAAN_MULTIMODAL_compact_results.csv


,experiment,split,ACC,BAL_ACC,SEN,SPE,AUC,F1,TP,TN,FP,FN
0,DAAN_MULTIMODAL_ADNI1_to_ADNI2,source_val,50.000000,50.000000,0.000000,100.000000,61.157025,0.000000,0,11,0,11
1,DAAN_MULTIMODAL_ADNI1_to_ADNI2,source_test,50.000000,50.000000,0.000000,100.000000,66.115702,0.000000,0,11,0,11
2,DAAN_MULTIMODAL_ADNI1_to_ADNI2,target_val_report,64.102564,50.000000,0.000000,100.000000,55.142857,0.000000,0,50,0,28
3,DAAN_MULTIMODAL_ADNI1_to_ADNI2,target_test_report,64.102564,50.000000,0.000000,100.000000,45.071429,0.000000,0,50,0,28
4,DAAN_MULTIMODAL_ADNI2_to_ADNI1,source_val,58.974359,61.714286,71.428571,52.000000,64.285714,55.555556,20,26,24,8
5,DAAN_MULTIMODAL_ADNI2_to_ADNI1,source_test,46.153846,51.714286,71.428571,32.000000,51.857143,48.780488,20,16,34,8
6,DAAN_MULTIMODAL_ADNI2_to_ADNI1,target_val_report,50.000000,50.000000,45.454545,54.545454,55.371901,47.619048,5,6,5,6
7,DAAN_MULTIMODAL_ADNI2_to_ADNI1,target_test_report,59.090909,59.090909,45.454545,72.727273,60.330579,52.631579,5,8,3,6
